In [1]:
import glob
import json
import re
import sys

import pandas as pd
import plotly.graph_objs as go
import plotly.io as pio

sys.path.insert(0, "../src/real_data")
from colors import color_mapping

In [2]:
YEAR = 2025
MODEL = "poisson_7"
NUM_GAMES = 380

BASE_PATH = f"../real_data/club_level_simulations/brazil/{YEAR}"
INPUT_PATH = f"{BASE_PATH}/inputs/poisson_data_{NUM_GAMES:03d}_games.json"
SAMPLES_DIR = f"{BASE_PATH}/{MODEL}/{NUM_GAMES}_games"

In [3]:
with open(INPUT_PATH, encoding="utf-8") as f:
    data = json.load(f)

team_mapping = {i + 1: name for i, name in enumerate(data["team_names"])}

csv_files = sorted(glob.glob(f"{SAMPLES_DIR}/{MODEL}-*.csv"))
samples = pd.concat(
    [pd.read_csv(f, comment="#") for f in csv_files],
    ignore_index=True,
)

samples = samples.drop(columns=[c for c in samples.columns if "raw" in c])
print(f"{len(csv_files)} chain files, {len(samples)} draws totais")

4 chain files, 4000 draws totais


In [4]:
samples.columns = [
    re.sub(r"^([a-z_]+)\.(\d+)$", r"\1[\2]", col) for col in samples.columns
]

n_clubs = len(team_mapping)
column_mapping = {}
for col in samples.columns:
    if "[" not in col:
        continue
    idx = int(col.split("[")[1].split("]")[0])
    if idx in team_mapping:
        column_mapping[col] = team_mapping[idx]

if len(column_mapping) == n_clubs:
    samples = samples.rename(columns=column_mapping)
elif len(column_mapping) == 2 * n_clubs:
    column_mapping = {
        k: v + (" (atk)" if "alpha" in k else " (def)")
        for k, v in column_mapping.items()
    }
    samples = samples.rename(columns=column_mapping)
    for team in team_mapping.values():
        samples[team] = samples[team + " (atk)"] - samples[team + " (def)"]
else:
    map_case = {
        "alpha": " (atk home)",
        "delta": " (atk away)",
        "gamma": " (def home)",
        "beta": " (def away)",
    }
    column_mapping = {
        k: v + map_case[k.split("[")[0]] for k, v in column_mapping.items()
    }
    samples = samples.rename(columns=column_mapping)
    for team in team_mapping.values():
        samples[team] = (
            samples[team + " (atk home)"] + samples[team + " (atk away)"]
        ) / 2 - (samples[team + " (def home)"] - samples[team + " (def away)"]) / 2

strengths = samples[list(team_mapping.values())]
strengths.head()

,São Paulo / SP,Sport / PE,Cruzeiro / MG,Mirassol / SP,Grêmio / RS,Atlético Mineiro / MG,Fortaleza / CE,Fluminense / RJ,Juventude / RS,Vitória / BA,Flamengo / RJ,Internacional / RS,Palmeiras / SP,Botafogo / RJ,Vasco da Gama / RJ,Santos / SP,Bahia / BA,Corinthians / SP,Red Bull Bragantino / SP,Ceará / CE
0,0.042825,-0.258245,0.269600,0.136219,-0.076777,-0.001070,-0.160022,0.174390,-0.301786,-0.167498,0.474098,-0.241389,0.128813,0.167465,-0.028796,0.136875,0.069367,-0.193906,-0.161700,-0.008463
1,0.072019,-0.213301,0.245285,0.165651,-0.037703,-0.019055,-0.188639,0.260678,-0.328455,-0.166483,0.372538,-0.147301,0.213967,0.158983,-0.013569,-0.043565,0.041880,-0.122784,-0.133495,-0.116652
2,-0.038115,-0.494044,0.270944,0.226798,0.048349,-0.086225,-0.176222,-0.018574,-0.439921,-0.265445,0.622337,-0.219103,0.522253,0.192415,-0.029922,-0.076607,0.043168,0.046247,-0.101640,-0.026693
3,-0.076713,-0.430607,0.256037,0.291381,-0.018696,0.077637,-0.088337,0.259980,-0.256133,-0.123313,0.362652,-0.088159,0.065595,0.227514,-0.023062,0.012141,-0.013033,-0.169254,-0.177206,-0.088425
4,-0.055970,-0.467527,0.179125,0.274963,-0.159918,0.068300,-0.154664,0.193491,-0.477731,-0.233393,0.480903,0.041827,0.204918,0.222913,-0.130867,0.104049,-0.005865,-0.078013,-0.055601,0.049060


In [5]:
samples_long = strengths.melt(var_name="Team", value_name="Strength")
team_means = samples_long.groupby("Team")["Strength"].mean().sort_values(ascending=True)
samples_long["Team"] = pd.Categorical(
    samples_long["Team"], categories=team_means.index, ordered=True
)

fig = go.Figure()
for team in team_means.index:
    team_data = samples_long[samples_long["Team"] == team]
    cor = color_mapping.get(team, "rgba(0,0,0,1)")
    cor_fill = cor.replace(",1)", ",0.35)")
    fig.add_trace(
        go.Violin(
            x=team_data["Strength"],
            y=team_data["Team"],
            name=team,
            orientation="h",
            side="positive",
            line_color=cor,
            fillcolor=cor_fill,
            line_width=1.2,
            meanline_visible=True,
            showlegend=False,
            scalemode="width",
            width=0.75,
            points=False,
        )
    )

fig.update_layout(
    height=500,
    width=700,
    title=f"Team Strengths - Serie A {YEAR} (after {NUM_GAMES} games)",
    title_font={"size": 14, "family": "Arial"},
    xaxis_title="Team Strength (log scale)",
    yaxis_title="Teams",
    xaxis_title_font={"size": 10},
    yaxis_title_font={"size": 10},
    xaxis_tickfont={"size": 9},
    margin={"l": 160, "r": 40, "t": 50, "b": 60},
    template="plotly_white",
    showlegend=False,
    yaxis={
        "categoryorder": "array",
        "categoryarray": team_means.index.tolist(),
        "tickfont": {"size": 9},
    },
)
fig.add_vline(x=0, line_dash="dash", line_color="red", line_width=1)

fig.show()

In [6]:
output_path = f"../master_text/figures/{YEAR}_team_strengths_violin.png"
pio.write_image(fig, output_path, format="png", scale=1, engine="kaleido")
print(f"Salvo em {output_path}")

Salvo em ../master_text/figures/2025_team_strengths_violin.png
